# Chapter 10.2: Scheduling Strategy Comparison

This notebook provides a deep comparative analysis of the scheduling strategies used by **vLLM** and **SGLang**. We build simulators, visualize scheduling decisions with Gantt charts, and analyze fairness and performance trade-offs.

## Learning Objectives
- Understand vLLM's SequenceGroup-based scheduling with preemption
- Understand SGLang's cache-aware, overlap scheduling
- Simulate and visualize both schedulers
- Analyze fairness and performance characteristics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part10/chapter_10.2_scheduling.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part10/chapter_10.2_scheduling.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

In [ ]:
# Setup
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import numpy as np
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from enum import Enum
import random
import heapq
from collections import deque

random.seed(42)
np.random.seed(42)
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11
print("Environment ready for scheduling comparison.")

## 1. Scheduling Strategy Overview

| Aspect | vLLM Scheduler | SGLang Scheduler |
|--------|---------------|------------------|
| **Unit** | SequenceGroup (request + all beams) | Individual Request (Req) |
| **Policy** | FCFS with priority-based preemption | Cache-aware with prefix locality |
| **Preemption** | Swap to CPU or Recompute | Evict from RadixCache (LRU) |
| **Batching** | Continuous batching (iteration-level) | Continuous batching with overlap |
| **Prefill/Decode** | Mixed in same batch (chunked prefill) | Can overlap prefill+decode |
| **Queue** | waiting -> running -> swapped | waiting -> running (simpler) |

In [ ]:
# Core data structures for simulation

class RequestPhase(Enum):
    WAITING = "waiting"
    PREFILL = "prefill"
    DECODE = "decode"
    PREEMPTED = "preempted"
    FINISHED = "finished"

@dataclass
class SimRequest:
    """A simulated inference request."""
    request_id: int
    arrival_time: float
    prompt_tokens: int
    output_tokens: int  # How many tokens to generate
    prefix_hash: str = ""  # For prefix matching
    
    # Runtime state
    phase: RequestPhase = RequestPhase.WAITING
    tokens_generated: int = 0
    blocks_allocated: int = 0
    start_time: float = 0.0
    end_time: float = 0.0
    preempt_count: int = 0
    wait_time: float = 0.0
    
    @property
    def total_tokens(self):
        return self.prompt_tokens + self.output_tokens
    
    @property
    def blocks_needed(self):
        """Blocks needed for current state (block_size=16)."""
        current_tokens = self.prompt_tokens + self.tokens_generated
        return (current_tokens + 15) // 16
    
    @property
    def is_done(self):
        return self.tokens_generated >= self.output_tokens

def generate_workload(n_requests: int, arrival_rate: float = 2.0,
                      prefix_groups: int = 3) -> List[SimRequest]:
    """Generate a realistic workload with shared prefixes."""
    requests = []
    prefixes = [f"prefix_{i}" for i in range(prefix_groups)]
    
    arrival = 0.0
    for i in range(n_requests):
        arrival += np.random.exponential(1.0 / arrival_rate)
        prompt_len = np.random.choice([64, 128, 256, 512], p=[0.3, 0.3, 0.25, 0.15])
        output_len = np.random.randint(20, 150)
        prefix = random.choice(prefixes)
        
        requests.append(SimRequest(
            request_id=i,
            arrival_time=arrival,
            prompt_tokens=prompt_len,
            output_tokens=output_len,
            prefix_hash=prefix
        ))
    return requests

workload = generate_workload(20)
print(f"Generated {len(workload)} requests")
print(f"{'ID':<5} {'Arrival':<10} {'Prompt':<10} {'Output':<10} {'Prefix':<15}")
print('-' * 50)
for r in workload[:10]:
    print(f"{r.request_id:<5} {r.arrival_time:<10.2f} {r.prompt_tokens:<10} {r.output_tokens:<10} {r.prefix_hash:<15}")
print(f"... and {len(workload)-10} more")

## 2. vLLM Scheduler Simulator

In [ ]:
# Demo 1: vLLM-style Scheduler

class VLLMSchedulerSimulator:
    """Simulates vLLM's SequenceGroup-based scheduler with preemption."""
    
    def __init__(self, total_blocks: int = 100, block_size: int = 16,
                 max_batch_tokens: int = 2048, max_batch_seqs: int = 16):
        self.total_blocks = total_blocks
        self.block_size = block_size
        self.max_batch_tokens = max_batch_tokens
        self.max_batch_seqs = max_batch_seqs
        self.free_blocks = total_blocks
        
        # Queues
        self.waiting: deque = deque()
        self.running: List[SimRequest] = []
        self.swapped: List[SimRequest] = []
        self.finished: List[SimRequest] = []
        
        # Scheduling log
        self.schedule_log = []  # (time, request_id, action, phase)
        self.time = 0.0
    
    def add_request(self, req: SimRequest):
        self.waiting.append(req)
    
    def _can_allocate(self, req: SimRequest) -> bool:
        return req.blocks_needed <= self.free_blocks
    
    def _allocate(self, req: SimRequest):
        blocks = req.blocks_needed
        self.free_blocks -= blocks
        req.blocks_allocated = blocks
    
    def _free(self, req: SimRequest):
        self.free_blocks += req.blocks_allocated
        req.blocks_allocated = 0
    
    def _preempt(self, req: SimRequest):
        """Preempt by swapping (move to CPU, free GPU blocks)."""
        self._free(req)
        req.phase = RequestPhase.PREEMPTED
        req.preempt_count += 1
        self.swapped.append(req)
        self.schedule_log.append((self.time, req.request_id, 'preempt', 'swapped'))
    
    def schedule_step(self) -> List[SimRequest]:
        """Run one scheduling step. Returns the batch to execute."""
        batch = []
        batch_tokens = 0
        
        # Step 1: Try to keep running requests going (decode)
        still_running = []
        for req in self.running:
            # Check if we need one more block for the next token
            new_blocks_needed = req.blocks_needed - req.blocks_allocated
            if new_blocks_needed > 0:
                if self.free_blocks >= new_blocks_needed:
                    self.free_blocks -= new_blocks_needed
                    req.blocks_allocated += new_blocks_needed
                else:
                    # Need to preempt - preempt last arrived (LIFO)
                    self._preempt(req)
                    continue
            still_running.append(req)
            batch.append(req)
            batch_tokens += 1  # decode = 1 token per request
        self.running = still_running
        
        # Step 2: Try to swap back preempted requests
        swap_back = []
        for req in self.swapped:
            if (len(batch) < self.max_batch_seqs and
                batch_tokens + 1 <= self.max_batch_tokens and
                self._can_allocate(req)):
                self._allocate(req)
                req.phase = RequestPhase.DECODE
                batch.append(req)
                batch_tokens += 1
                self.running.append(req)
                swap_back.append(req)
                self.schedule_log.append((self.time, req.request_id, 'swap_in', 'decode'))
        for r in swap_back:
            self.swapped.remove(r)
        
        # Step 3: Admit new requests from waiting (FCFS)
        while self.waiting:
            req = self.waiting[0]
            if req.arrival_time > self.time:
                break
            if (len(batch) >= self.max_batch_seqs or
                batch_tokens + req.prompt_tokens > self.max_batch_tokens):
                break
            if not self._can_allocate(req):
                break
            
            self.waiting.popleft()
            self._allocate(req)
            req.phase = RequestPhase.PREFILL
            req.start_time = self.time
            req.wait_time = self.time - req.arrival_time
            batch.append(req)
            batch_tokens += req.prompt_tokens
            self.running.append(req)
            self.schedule_log.append((self.time, req.request_id, 'admit', 'prefill'))
        
        return batch
    
    def execute_step(self, batch: List[SimRequest]):
        """Simulate execution of one batch step."""
        for req in batch:
            if req.phase == RequestPhase.PREFILL:
                req.phase = RequestPhase.DECODE
                self.schedule_log.append((self.time, req.request_id, 'prefill_done', 'decode'))
            else:
                req.tokens_generated += 1
                if req.is_done:
                    req.phase = RequestPhase.FINISHED
                    req.end_time = self.time
                    self._free(req)
                    self.running.remove(req)
                    self.finished.append(req)
                    self.schedule_log.append((self.time, req.request_id, 'finish', 'done'))
    
    def run(self, requests: List[SimRequest], max_steps: int = 500) -> Dict:
        """Run the full simulation."""
        import copy
        reqs = [copy.deepcopy(r) for r in requests]
        for r in reqs:
            self.add_request(r)
        
        step = 0
        self.time = 0.0
        dt = 0.05  # time per step
        
        while step < max_steps and (self.waiting or self.running or self.swapped):
            batch = self.schedule_step()
            if batch:
                self.execute_step(batch)
            self.time += dt
            step += 1
        
        return {
            'finished': len(self.finished),
            'total_time': self.time,
            'avg_latency': np.mean([r.end_time - r.arrival_time for r in self.finished]) if self.finished else 0,
            'avg_wait': np.mean([r.wait_time for r in self.finished]) if self.finished else 0,
            'preemptions': sum(r.preempt_count for r in self.finished),
            'schedule_log': self.schedule_log,
            'requests': self.finished,
        }

# Run vLLM scheduler simulation
vllm_sched = VLLMSchedulerSimulator(total_blocks=60, max_batch_seqs=8)
vllm_results = vllm_sched.run(workload)
print(f"vLLM Scheduler Results:")
print(f"  Completed: {vllm_results['finished']} / {len(workload)} requests")
print(f"  Total time: {vllm_results['total_time']:.2f}s")
print(f"  Avg latency: {vllm_results['avg_latency']:.2f}s")
print(f"  Avg wait time: {vllm_results['avg_wait']:.2f}s")
print(f"  Total preemptions: {vllm_results['preemptions']}")

In [ ]:
# Demo 2: SGLang-style Scheduler

class SGLangSchedulerSimulator:
    """Simulates SGLang's cache-aware scheduler with RadixCache."""
    
    def __init__(self, total_blocks: int = 100, block_size: int = 16,
                 max_batch_tokens: int = 2048, max_batch_seqs: int = 16):
        self.total_blocks = total_blocks
        self.block_size = block_size
        self.max_batch_tokens = max_batch_tokens
        self.max_batch_seqs = max_batch_seqs
        self.free_blocks = total_blocks
        
        # Cache: prefix_hash -> cached_blocks (simulated radix cache)
        self.prefix_cache: Dict[str, int] = {}
        
        # Queues
        self.waiting: List[SimRequest] = []
        self.running: List[SimRequest] = []
        self.finished: List[SimRequest] = []
        
        self.schedule_log = []
        self.time = 0.0
        self.cache_hits = 0
        self.cache_misses = 0
    
    def add_request(self, req: SimRequest):
        self.waiting.append(req)
    
    def _get_prefix_savings(self, req: SimRequest) -> int:
        """Check radix cache for prefix match."""
        if req.prefix_hash in self.prefix_cache:
            cached = self.prefix_cache[req.prefix_hash]
            return min(cached, req.prompt_tokens)  # Token-level match
        return 0
    
    def _cache_aware_sort(self):
        """Sort waiting queue by cache locality (prefix match first)."""
        def sort_key(req):
            savings = self._get_prefix_savings(req)
            # Prioritize: more cache savings first, then earlier arrival
            return (-savings, req.arrival_time)
        self.waiting.sort(key=sort_key)
    
    def schedule_step(self) -> List[SimRequest]:
        """Cache-aware scheduling step."""
        batch = []
        batch_tokens = 0
        
        # Step 1: Keep running requests (decode)
        for req in self.running:
            new_blocks_needed = req.blocks_needed - req.blocks_allocated
            if new_blocks_needed > 0:
                if self.free_blocks >= new_blocks_needed:
                    self.free_blocks -= new_blocks_needed
                    req.blocks_allocated += new_blocks_needed
                else:
                    # SGLang: evict from cache instead of preempting running requests
                    self._evict_cache(new_blocks_needed)
                    if self.free_blocks >= new_blocks_needed:
                        self.free_blocks -= new_blocks_needed
                        req.blocks_allocated += new_blocks_needed
            batch.append(req)
            batch_tokens += 1
        
        # Step 2: Sort waiting by cache locality
        self._cache_aware_sort()
        
        # Step 3: Admit new requests
        new_waiting = []
        for req in self.waiting:
            if req.arrival_time > self.time:
                new_waiting.append(req)
                continue
            
            prefix_savings = self._get_prefix_savings(req)
            effective_prefill = req.prompt_tokens - prefix_savings
            blocks_needed = (req.prompt_tokens + 15) // 16  # Still need all blocks
            cached_blocks = prefix_savings // 16  # But some are already cached
            new_blocks = blocks_needed - cached_blocks
            
            if (len(batch) >= self.max_batch_seqs or
                batch_tokens + effective_prefill > self.max_batch_tokens or
                self.free_blocks < new_blocks):
                new_waiting.append(req)
                continue
            
            # Admit
            self.free_blocks -= new_blocks
            req.blocks_allocated = blocks_needed
            req.phase = RequestPhase.PREFILL
            req.start_time = self.time
            req.wait_time = self.time - req.arrival_time
            batch.append(req)
            batch_tokens += effective_prefill
            self.running.append(req)
            
            if prefix_savings > 0:
                self.cache_hits += 1
                self.schedule_log.append((self.time, req.request_id, 'admit_cached',
                                        f'prefill (saved {prefix_savings} tokens)'))
            else:
                self.cache_misses += 1
                self.schedule_log.append((self.time, req.request_id, 'admit', 'prefill'))
        
        self.waiting = new_waiting
        return batch
    
    def _evict_cache(self, blocks_needed: int):
        """LRU eviction from radix cache."""
        while self.free_blocks < blocks_needed and self.prefix_cache:
            # Evict least recently used prefix
            oldest = min(self.prefix_cache, key=lambda k: self.prefix_cache[k])
            freed = self.prefix_cache.pop(oldest)
            self.free_blocks += freed // 16
    
    def execute_step(self, batch: List[SimRequest]):
        for req in batch:
            if req.phase == RequestPhase.PREFILL:
                req.phase = RequestPhase.DECODE
                # Update radix cache with this prefix
                self.prefix_cache[req.prefix_hash] = req.prompt_tokens
            else:
                req.tokens_generated += 1
                if req.is_done:
                    req.phase = RequestPhase.FINISHED
                    req.end_time = self.time
                    # Don't immediately free - keep in cache!
                    # (Only free blocks beyond cached prefix)
                    output_blocks = (req.output_tokens + 15) // 16
                    self.free_blocks += output_blocks
                    self.running.remove(req)
                    self.finished.append(req)
                    self.schedule_log.append((self.time, req.request_id, 'finish', 'done'))
    
    def run(self, requests: List[SimRequest], max_steps: int = 500) -> Dict:
        import copy
        reqs = [copy.deepcopy(r) for r in requests]
        for r in reqs:
            self.add_request(r)
        
        step = 0
        self.time = 0.0
        dt = 0.05
        
        while step < max_steps and (self.waiting or self.running):
            batch = self.schedule_step()
            if batch:
                self.execute_step(batch)
            self.time += dt
            step += 1
        
        return {
            'finished': len(self.finished),
            'total_time': self.time,
            'avg_latency': np.mean([r.end_time - r.arrival_time for r in self.finished]) if self.finished else 0,
            'avg_wait': np.mean([r.wait_time for r in self.finished]) if self.finished else 0,
            'preemptions': 0,  # SGLang doesn't preempt running requests
            'cache_hits': self.cache_hits,
            'cache_misses': self.cache_misses,
            'schedule_log': self.schedule_log,
            'requests': self.finished,
        }

# Run SGLang scheduler simulation
sglang_sched = SGLangSchedulerSimulator(total_blocks=60, max_batch_seqs=8)
sglang_results = sglang_sched.run(workload)
print(f"SGLang Scheduler Results:")
print(f"  Completed: {sglang_results['finished']} / {len(workload)} requests")
print(f"  Total time: {sglang_results['total_time']:.2f}s")
print(f"  Avg latency: {sglang_results['avg_latency']:.2f}s")
print(f"  Avg wait time: {sglang_results['avg_wait']:.2f}s")
print(f"  Cache hits: {sglang_results['cache_hits']}, misses: {sglang_results['cache_misses']}")

In [ ]:
# Demo 3: Side-by-side results comparison

def compare_results(vllm_res, sglang_res):
    metrics = ['Completed', 'Total Time (s)', 'Avg Latency (s)',
               'Avg Wait (s)', 'Preemptions']
    vllm_vals = [vllm_res['finished'], f"{vllm_res['total_time']:.2f}",
                 f"{vllm_res['avg_latency']:.2f}", f"{vllm_res['avg_wait']:.2f}",
                 vllm_res['preemptions']]
    sglang_vals = [sglang_res['finished'], f"{sglang_res['total_time']:.2f}",
                   f"{sglang_res['avg_latency']:.2f}", f"{sglang_res['avg_wait']:.2f}",
                   sglang_res.get('preemptions', 0)]
    
    print(f"{'Metric':<25} {'vLLM':<15} {'SGLang':<15} {'Winner':<10}")
    print('=' * 65)
    for m, v, s in zip(metrics, vllm_vals, sglang_vals):
        # Determine winner (lower is better for most metrics)
        try:
            vf, sf = float(v), float(s)
            if m == 'Completed':
                winner = 'vLLM' if vf > sf else 'SGLang' if sf > vf else 'Tie'
            else:
                winner = 'vLLM' if vf < sf else 'SGLang' if sf < vf else 'Tie'
        except ValueError:
            winner = '-'
        print(f"{m:<25} {str(v):<15} {str(s):<15} {winner:<10}")

compare_results(vllm_results, sglang_results)

In [ ]:
# Demo 4: Gantt Chart visualization of scheduling decisions

def plot_gantt_charts(vllm_res, sglang_res, max_requests=12):
    """Visualize scheduling decisions as Gantt charts."""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True)
    
    phase_colors = {
        'prefill': '#FF9800',
        'decode': '#4CAF50',
        'preempted': '#F44336',
        'waiting': '#9E9E9E',
        'swap_in': '#2196F3',
    }
    
    def plot_schedule(ax, results, title, color_accent):
        reqs = sorted(results['requests'][:max_requests], key=lambda r: r.request_id)
        
        for idx, req in enumerate(reqs):
            # Waiting period
            if req.wait_time > 0:
                ax.barh(idx, req.wait_time, left=req.arrival_time,
                       height=0.6, color='#E0E0E0', edgecolor='gray', alpha=0.5)
            
            # Prefill
            prefill_duration = 0.05 * (req.prompt_tokens / 64)  # Proportional
            ax.barh(idx, prefill_duration, left=req.start_time,
                   height=0.6, color='#FF9800', edgecolor='black', linewidth=0.5)
            
            # Decode
            decode_duration = req.end_time - req.start_time - prefill_duration
            if decode_duration > 0:
                ax.barh(idx, decode_duration, left=req.start_time + prefill_duration,
                       height=0.6, color='#4CAF50', edgecolor='black', linewidth=0.5)
            
            # Preemption markers
            if req.preempt_count > 0:
                ax.plot(req.start_time + prefill_duration, idx, 'rv', markersize=8)
        
        ax.set_yticks(range(len(reqs)))
        ax.set_yticklabels([f'Req {r.request_id}' for r in reqs], fontsize=9)
        ax.set_title(title, fontsize=14, fontweight='bold', color=color_accent)
        ax.set_xlabel('Time (s)')
        ax.grid(axis='x', alpha=0.3)
    
    plot_schedule(ax1, vllm_res, 'vLLM Scheduler - Gantt Chart', '#2196F3')
    plot_schedule(ax2, sglang_res, 'SGLang Scheduler - Gantt Chart', '#4CAF50')
    
    # Legend
    legend_patches = [
        mpatches.Patch(color='#E0E0E0', label='Waiting'),
        mpatches.Patch(color='#FF9800', label='Prefill'),
        mpatches.Patch(color='#4CAF50', label='Decode'),
        plt.Line2D([0], [0], marker='v', color='red', label='Preemption', linestyle='None'),
    ]
    fig.legend(handles=legend_patches, loc='upper right', ncol=4, fontsize=10)
    
    plt.tight_layout()
    plt.savefig('gantt_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_gantt_charts(vllm_results, sglang_results)

In [ ]:
# Demo 5: Preemption strategy comparison

print("=" * 70)
print("Preemption Strategy Comparison")
print("=" * 70)

strategies = {
    'vLLM: Swap': {
        'mechanism': 'Copy KV cache blocks from GPU to CPU RAM',
        'cost': 'High (PCIe bandwidth limited, ~12 GB/s)',
        'recovery': 'Swap back from CPU when GPU blocks available',
        'recovery_cost': 'Same as swap out',
        'when_used': 'GPU memory pressure + enough CPU memory',
    },
    'vLLM: Recompute': {
        'mechanism': 'Discard KV cache, recompute prefill on resume',
        'cost': 'None (instant eviction)',
        'recovery': 'Full recomputation of prefill',
        'recovery_cost': 'High (proportional to prompt length)',
        'when_used': 'Short prompts or when CPU memory insufficient',
    },
    'SGLang: Cache Eviction': {
        'mechanism': 'Evict cached prefixes from RadixCache (LRU)',
        'cost': 'None (just pointer removal)',
        'recovery': 'Prefix must be recomputed on next similar request',
        'recovery_cost': 'Amortized across future requests',
        'when_used': 'GPU memory pressure (never preempts running requests)',
    },
}

for name, details in strategies.items():
    print(f"\n{name}:")
    for k, v in details.items():
        print(f"  {k}: {v}")

# Visualize preemption cost
fig, ax = plt.subplots(figsize=(10, 5))
strategies_list = ['vLLM Swap', 'vLLM Recompute', 'SGLang Cache Evict']
eviction_cost = [8, 1, 0.5]  # Relative cost units
recovery_cost = [8, 15, 3]   # Relative cost units

x = np.arange(len(strategies_list))
width = 0.35
ax.bar(x - width/2, eviction_cost, width, label='Eviction Cost', color='#F44336', alpha=0.8)
ax.bar(x + width/2, recovery_cost, width, label='Recovery Cost', color='#2196F3', alpha=0.8)
ax.set_ylabel('Relative Cost (arbitrary units)')
ax.set_title('Preemption Strategy Costs', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(strategies_list)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Demo 6: Fairness Analysis

def analyze_fairness(vllm_res, sglang_res):
    """Compare fairness: how evenly are latencies distributed?"""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    vllm_latencies = [r.end_time - r.arrival_time for r in vllm_res['requests']]
    sglang_latencies = [r.end_time - r.arrival_time for r in sglang_res['requests']]
    
    # 1. Latency Distribution
    axes[0].hist(vllm_latencies, bins=15, alpha=0.6, color='#2196F3', label='vLLM', edgecolor='black')
    axes[0].hist(sglang_latencies, bins=15, alpha=0.6, color='#4CAF50', label='SGLang', edgecolor='black')
    axes[0].set_xlabel('Latency (s)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Latency Distribution', fontweight='bold')
    axes[0].legend()
    
    # 2. Latency vs Arrival Time (fairness over time)
    vllm_arrivals = [r.arrival_time for r in vllm_res['requests']]
    sglang_arrivals = [r.arrival_time for r in sglang_res['requests']]
    
    axes[1].scatter(vllm_arrivals, vllm_latencies, alpha=0.7, color='#2196F3',
                   label='vLLM', s=50, edgecolor='black')
    axes[1].scatter(sglang_arrivals, sglang_latencies, alpha=0.7, color='#4CAF50',
                   label='SGLang', s=50, marker='s', edgecolor='black')
    axes[1].set_xlabel('Arrival Time (s)')
    axes[1].set_ylabel('Latency (s)')
    axes[1].set_title('Latency vs Arrival (Fairness)', fontweight='bold')
    axes[1].legend()
    
    # 3. CDF of latencies
    for latencies, label, color in [(vllm_latencies, 'vLLM', '#2196F3'),
                                     (sglang_latencies, 'SGLang', '#4CAF50')]:
        sorted_lat = np.sort(latencies)
        cdf = np.arange(1, len(sorted_lat) + 1) / len(sorted_lat)
        axes[2].plot(sorted_lat, cdf, '-o', color=color, label=label, markersize=4)
    axes[2].set_xlabel('Latency (s)')
    axes[2].set_ylabel('CDF')
    axes[2].set_title('Latency CDF', fontweight='bold')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Fairness metrics
    print(f"\nFairness Metrics:")
    print(f"  {'Metric':<30} {'vLLM':<15} {'SGLang':<15}")
    print(f"  {'-'*60}")
    print(f"  {'Latency Std Dev':<30} {np.std(vllm_latencies):<15.3f} {np.std(sglang_latencies):<15.3f}")
    print(f"  {'Max/Min Latency Ratio':<30} {max(vllm_latencies)/max(min(vllm_latencies),0.01):<15.3f} {max(sglang_latencies)/max(min(sglang_latencies),0.01):<15.3f}")
    print(f"  {'Jain Fairness Index':<30} ", end='')
    
    def jain_index(latencies):
        n = len(latencies)
        return sum(latencies)**2 / (n * sum(x**2 for x in latencies))
    
    print(f"{jain_index(vllm_latencies):<15.4f} {jain_index(sglang_latencies):<15.4f}")
    print(f"  (Jain index closer to 1.0 = more fair)")

analyze_fairness(vllm_results, sglang_results)

In [ ]:
# Demo 7: Batch utilization over time

def plot_batch_utilization(vllm_sched, sglang_sched, workload):
    """Track and plot batch size over time for both schedulers."""
    import copy
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    for sched_class, ax, title, color in [
        (VLLMSchedulerSimulator, ax1, 'vLLM Batch Size Over Time', '#2196F3'),
        (SGLangSchedulerSimulator, ax2, 'SGLang Batch Size Over Time', '#4CAF50')
    ]:
        sched = sched_class(total_blocks=60, max_batch_seqs=8)
        reqs = [copy.deepcopy(r) for r in workload]
        for r in reqs:
            sched.add_request(r)
        
        times = []
        batch_sizes = []
        free_blocks_over_time = []
        
        sched.time = 0.0
        dt = 0.05
        for _ in range(500):
            batch = sched.schedule_step()
            if batch:
                sched.execute_step(batch)
            times.append(sched.time)
            batch_sizes.append(len(batch))
            free_blocks_over_time.append(sched.free_blocks)
            sched.time += dt
            if not (sched.waiting or sched.running or getattr(sched, 'swapped', [])):
                break
        
        ax.fill_between(times, batch_sizes, alpha=0.3, color=color)
        ax.plot(times, batch_sizes, color=color, linewidth=1.5)
        ax.set_ylabel('Batch Size')
        ax.set_title(title, fontweight='bold', color=color)
        ax.set_ylim(0, 10)
        ax.grid(True, alpha=0.3)
        
        # Secondary axis for free blocks
        ax_twin = ax.twinx()
        ax_twin.plot(times, free_blocks_over_time, color='red', alpha=0.5,
                    linewidth=1, linestyle='--', label='Free blocks')
        ax_twin.set_ylabel('Free Blocks', color='red')
        ax_twin.tick_params(axis='y', labelcolor='red')
    
    ax2.set_xlabel('Time (s)')
    plt.tight_layout()
    plt.show()

plot_batch_utilization(VLLMSchedulerSimulator, SGLangSchedulerSimulator, workload)

---
## Hands-on Assignment 1: Build a Scheduler Simulator for Both Systems

Extend the simulators above to support:
1. **Chunked prefill** (vLLM): split long prefills into chunks to reduce HoL blocking
2. **Overlap scheduling** (SGLang): schedule next prefill while current decode is running
3. Compare results with and without these features

In [ ]:
# Assignment 1: Extend both schedulers

class VLLMChunkedPrefillScheduler(VLLMSchedulerSimulator):
    """vLLM scheduler with chunked prefill support."""
    
    def __init__(self, *args, chunk_size: int = 512, **kwargs):
        super().__init__(*args, **kwargs)
        self.chunk_size = chunk_size
    
    def schedule_step(self) -> List[SimRequest]:
        # TODO: Implement chunked prefill scheduling
        # Key changes:
        # 1. When admitting a new request with prompt > chunk_size,
        #    only process chunk_size tokens in this step
        # 2. Keep the request in a "partial_prefill" state
        # 3. Allow decode requests to run alongside partial prefills
        
        # Hint: track req.prefill_progress and compare to req.prompt_tokens
        return super().schedule_step()  # Replace with your implementation


class SGLangOverlapScheduler(SGLangSchedulerSimulator):
    """SGLang scheduler with overlap scheduling."""
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
    
    def schedule_step(self) -> List[SimRequest]:
        # TODO: Implement overlap scheduling
        # Key changes:
        # 1. While decode batch is running on GPU, prepare next prefill batch
        # 2. Overlap CPU tokenization with GPU execution
        # 3. Use "extend" to mix prefill and decode in same forward pass
        
        return super().schedule_step()  # Replace with your implementation


# Test your implementations
print("Test your chunked prefill and overlap scheduling implementations.")
print("Compare results with the basic schedulers above.")
print("\nExpected improvements:")
print("  - Chunked prefill: Lower P99 latency, better fairness")
print("  - Overlap scheduling: Higher throughput, better GPU utilization")

---
## Hands-on Assignment 2: Find Workloads Where Each Scheduler Wins

Design specific workloads that favor one scheduler over the other.

In [ ]:
# Assignment 2: Design targeted workloads

def create_workload_vllm_wins():
    """Create a workload where vLLM's scheduler performs better.
    
    Hint: vLLM should win when:
    - Requests have unique prompts (no prefix sharing benefit)
    - High memory pressure requires efficient preemption
    - Uniform short prompts benefit from simple FCFS
    """
    requests = []
    # TODO: Generate 20 requests that favor vLLM's approach
    for i in range(20):
        requests.append(SimRequest(
            request_id=i,
            arrival_time=i * 0.3,
            prompt_tokens=64,  # TODO: choose parameters that favor vLLM
            output_tokens=50,
            prefix_hash=f"unique_{i}"  # All unique prefixes
        ))
    return requests

def create_workload_sglang_wins():
    """Create a workload where SGLang's scheduler performs better.
    
    Hint: SGLang should win when:
    - Many requests share prefixes (RadixCache advantage)
    - Multi-turn conversations (prefix reuse)
    - Bursty arrival patterns with shared system prompts
    """
    requests = []
    # TODO: Generate 20 requests that favor SGLang's approach
    system_prompt = "shared_system"  # Many requests share this
    for i in range(20):
        requests.append(SimRequest(
            request_id=i,
            arrival_time=i * 0.2,
            prompt_tokens=256,  # TODO: choose parameters that favor SGLang
            output_tokens=80,
            prefix_hash=system_prompt  # Shared prefix!
        ))
    return requests

# Run both workloads through both schedulers
import copy

for workload_name, workload_fn in [('vLLM-favorable', create_workload_vllm_wins),
                                     ('SGLang-favorable', create_workload_sglang_wins)]:
    wl = workload_fn()
    print(f"\n{'='*60}")
    print(f"Workload: {workload_name}")
    print(f"{'='*60}")
    
    v_sched = VLLMSchedulerSimulator(total_blocks=60, max_batch_seqs=8)
    v_res = v_sched.run(wl)
    
    s_sched = SGLangSchedulerSimulator(total_blocks=60, max_batch_seqs=8)
    s_res = s_sched.run(wl)
    
    compare_results(v_res, s_res)

---
## Hands-on Assignment 3: Design a Hybrid Scheduler

Design a scheduler that combines the best of both approaches:
- Cache-aware scheduling from SGLang
- Preemption capabilities from vLLM
- Chunked prefill for fairness

In [ ]:
# Assignment 3: Hybrid Scheduler

class HybridScheduler:
    """A scheduler combining vLLM and SGLang approaches.
    
    Design goals:
    1. Cache-aware request ordering (from SGLang)
    2. Preemption with swap (from vLLM) as fallback
    3. Chunked prefill for fairness
    4. Overlap scheduling for throughput
    """
    
    def __init__(self, total_blocks: int = 100, max_batch_seqs: int = 16,
                 chunk_size: int = 512):
        self.total_blocks = total_blocks
        self.max_batch_seqs = max_batch_seqs
        self.chunk_size = chunk_size
        self.free_blocks = total_blocks
        
        # TODO: Initialize data structures
        # You need:
        # - A waiting queue
        # - A running list
        # - A swapped list (from vLLM)
        # - A prefix cache (from SGLang)
        self.waiting = []
        self.running = []
        self.swapped = []
        self.prefix_cache = {}
        self.finished = []
        self.time = 0.0
        self.schedule_log = []
    
    def schedule_step(self):
        """Implement your hybrid scheduling algorithm.
        
        Suggested approach:
        1. First, handle running requests (decode)
        2. If memory pressure, try cache eviction first (SGLang-style)
        3. If still pressure, preempt lowest-priority request (vLLM-style)
        4. Sort waiting queue by cache locality
        5. Admit new requests with chunked prefill
        """
        # TODO: Implement the hybrid scheduling logic
        pass
    
    def run(self, requests, max_steps=500):
        # TODO: Implement the run loop
        pass

print("Implement the HybridScheduler and compare it against both baseline schedulers.")
print("\nEvaluation criteria:")
print("  1. Throughput (requests/second)")
print("  2. Average latency")
print("  3. P99 latency")
print("  4. Fairness (Jain index)")
print("  5. Cache hit rate")
print("\nYour hybrid should ideally beat both baselines on at least 3 of 5 metrics.")

In [ ]:
# Assignment 3 (continued): Evaluation template

def evaluate_schedulers(workload, schedulers_dict):
    """Run all schedulers on the same workload and compare."""
    import copy
    
    results = {}
    for name, sched_class in schedulers_dict.items():
        sched = sched_class(total_blocks=60, max_batch_seqs=8)
        res = sched.run(copy.deepcopy(workload))
        results[name] = res
    
    # Compare
    print(f"{'Metric':<25}", end='')
    for name in results:
        print(f"{name:<20}", end='')
    print()
    print('=' * (25 + 20 * len(results)))
    
    metrics = [
        ('Completed', lambda r: r['finished']),
        ('Avg Latency', lambda r: f"{r['avg_latency']:.3f}"),
        ('Avg Wait', lambda r: f"{r['avg_wait']:.3f}"),
        ('Preemptions', lambda r: r.get('preemptions', 0)),
    ]
    
    for metric_name, metric_fn in metrics:
        print(f"{metric_name:<25}", end='')
        for name, res in results.items():
            print(f"{str(metric_fn(res)):<20}", end='')
        print()

# Example usage (uncomment when HybridScheduler is implemented)
# evaluate_schedulers(workload, {
#     'vLLM': VLLMSchedulerSimulator,
#     'SGLang': SGLangSchedulerSimulator,
#     'Hybrid': HybridScheduler,
# })

print("\nUncomment the evaluation code above after implementing HybridScheduler.")
print("\n--- End of Chapter 10.2 ---")
print("Next: Chapter 10.3 - Memory Management Comparison")

## Summary

| Topic | Key Insight |
|-------|-------------|
| **vLLM Scheduling** | FCFS with SequenceGroup-based preemption (swap/recompute) |
| **SGLang Scheduling** | Cache-aware ordering prioritizes prefix locality |
| **Preemption** | vLLM preempts running requests; SGLang evicts cached data |
| **Fairness** | vLLM's FCFS is inherently fair; SGLang may reorder for cache hits |
| **Overlap** | SGLang can overlap prefill+decode; vLLM uses chunked prefill |